In [ ]:
%reload_ext autoreload
%autoreload 2
import uproot
import pandas as pd
import numpy as np

def load_root_to_dataframe(file_path):
    """
    Opens a ROOT file using uproot and loads its content into a pandas DataFrame.
    
    Args:
        file_path (str): Path to the ROOT file
        
    Returns:
        pd.DataFrame: DataFrame containing the ROOT file data
    """
    with uproot.open(file_path) as file:
        # Get the first tree in the file
        tree = file[file.keys()[0]]
        
        # Convert tree to DataFrame
        df = tree.arrays(library="pd")
        
    return df


def is_scalar_series(series: pd.Series) -> bool:
    """Return True if a series appears to store scalar values per row."""
    non_null = series.dropna()
    if non_null.empty:
        return True
    return bool(np.isscalar(non_null.iloc[0]))

In [3]:

file_path = "/user/abiolchi/muraves_outputs/RECONSTRUCTED/NERO/MURAVES_AnalyzedData_run2546_py.root"  # Replace with your actual file path
df = load_root_to_dataframe(file_path)
print(df.head())
 

    Run  Nclusters_Z1  Nclusters_Z2  Nclusters_Z3  Nclusters_Z4  Nclusters_Y1  \
0  2546             1             1             1             1             2   
1  2546             0             1             2             0             1   
2  2546             0             1             2             0             1   
3  2546             1             2             0             1             2   
4  2546             1             1             2             0             1   

   Nclusters_Y2  Nclusters_Y3  Nclusters_Y4  nClusterSize_Z1  ...  \
0             1             1             0                1  ...   
1             1             1             1                0  ...   
2             1             1             0                0  ...   
3             1             1             1                1  ...   
4             2             1             1                1  ...   

      timestamp  WorkingPoint  Temperature  TriggerRate  nTriggerMaskChannels  \
0  1.572315e+12  

In [ ]:
FEATURES_LABEL = {
        "Nclusters_Y1": "Nclusters_Y1",
        "Nclusters_Y2": "Nclusters_Y2",
        "Nclusters_Y3": "Nclusters_Y3",
        "Nclusters_Y4": "Nclusters_Y4",
        "Nclusters_Z1": "Nclusters_Z1",
        "Nclusters_Z2": "Nclusters_Z2",
        "Nclusters_Z3": "Nclusters_Z3",
        "Nclusters_Z4": "Nclusters_Z4",
        "StripsID_Y1": "StripsID_Y1",
        "StripsID_Y2": "StripsID_Y2",
        "StripsID_Y3": "StripsID_Y3",
        "StripsID_Y4": "StripsID_Y4",
        "StripsID_Z1": "StripsID_Z1",
        "StripsID_Z2": "StripsID_Z2",
        "StripsID_Z3": "StripsID_Z3",
        "StripsID_Z4": "StripsID_Z4",
        "Ntracks_3p_xy": "Ntracks_3p_xy",
        "Ntracks_3p_xz": "Ntracks_3p_xz",
        "nStripsPosition_Y1": "nStripsPosition_Y1",
        "nStripsPosition_Y2": "nStripsPosition_Y2",
        "nStripsPosition_Y3": "nStripsPosition_Y3",
        "nStripsPosition_Y4": "nStripsPosition_Y4",
        "nStripsPosition_Z1": "nStripsPosition_Z1",
        "nStripsPosition_Z2": "nStripsPosition_Z2",
        "nStripsPosition_Z3": "nStripsPosition_Z3",
        "nStripsPosition_Z4": "nStripsPosition_Z4",
        "chiSquare_3p_xy": "chiSquare_3p_xy",
        "chiSquare_3p_xz": "chiSquare_3p_xz",
        "chiSquare_4p_xy": "chiSquare_4p_xy",
        "chiSquare_4p_xz": "chiSquare_4p_xz",
        "TriggerMaskChannels": "TriggerMaskChannels",
        "TriggerMaskSize": "TriggerMaskSize",
        "TriggerMaskStrips": "TriggerMaskStrips",
    }

candidate_features = [col for col in df.columns if col in FEATURES_LABEL.keys()]
FEATURES = [
    col for col in candidate_features
    if pd.api.types.is_numeric_dtype(df[col]) and is_scalar_series(df[col])
]

dropped = sorted(set(candidate_features) - set(FEATURES))
if dropped:
    print("Dropping non-scalar features for PCA:", ", ".join(dropped))

if not FEATURES:
    raise ValueError("No scalar numeric features available for PCA")

try:
    from dev.pca_analysis.pca_mapper import PCAMapper
except ModuleNotFoundError:
    from pca_mapper import PCAMapper

pca_mapper = PCAMapper(
    raw_df=df,
    features=FEATURES,
    feature_labels=FEATURES_LABEL,
).fit()

print("PCA fit completed with", len(FEATURES), "features")

TypeError: 'NoneType' object is not iterable